In [28]:
!pip install transformers torch -q


In [34]:
import torch
import time
from transformers import AutoModelForCausalLM, AutoTokenizer


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [31]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)


tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

print("Model loaded successfully!")


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [37]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day!")
        break

    # Encode input
    new_input = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt').to(device)
    new_input_ids = new_input["input_ids"]
    attention_mask = new_input["attention_mask"]

    # Maintain chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.cat(
            [torch.ones(chat_history_ids.shape, dtype=torch.long).to(device), attention_mask],
            dim=-1
        )
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=500,
        attention_mask=attention_mask,

        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    response = response.strip()
    time.sleep(1)

    print("Chatbot:", response.strip())


Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: hi
Chatbot: Hi! How are you?
You: i'm good, what about you
Chatbot: I'm doing good too!
You: what AI in technolog
Chatbot: I'm in the middle of studying.
You: quit
Chatbot: Goodbye! Have a great day!
